### Этап 1: Инициализация и очистка

In [282]:
# Импорт необходимых модулей и подключение к базе данных, чтение очищенного представления(view)

import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
conn = sqlite3.connect('../SQL/Kickstarter_db.db')
df = pd.read_sql_query("SELECT * FROM KS_Projects_Modified", conn)
conn.close()

In [283]:
# Преобразование дат в корректный формат

df['launched'] = pd.to_datetime(df['launched'])
df['deadline'] = pd.to_datetime(df['deadline'])

In [284]:
# Проверка ID на дубликаты

print("ID duplicates: ",df['ID'].duplicated().sum())

ID duplicates:  0


In [285]:
# Введение вспомогательных столбцов

df['is_successful'] = df['state'] == 'successful'
df['duration'] = (df['deadline'] - df['launched']).dt.days

In [286]:
# Получаем общую информацию о датафрейме

df.info()
df.sample(5)

<class 'pandas.DataFrame'>
RangeIndex: 319321 entries, 0 to 319320
Data columns (total 15 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ID             319321 non-null  int64         
 1   name           319321 non-null  str           
 2   category       319321 non-null  str           
 3   main_category  319321 non-null  str           
 4   currency       319321 non-null  str           
 5   deadline       319321 non-null  datetime64[us]
 6   goal           319321 non-null  float64       
 7   launched       319321 non-null  datetime64[us]
 8   pledged        319321 non-null  float64       
 9   state          319321 non-null  str           
 10  backers        319321 non-null  int64         
 11  country        319321 non-null  str           
 12  usd pledged    319321 non-null  float64       
 13  is_successful  319321 non-null  bool          
 14  duration       319321 non-null  int64         
dtypes: bool(1),

,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged,is_successful,duration
231787,467561054,Fly the Skigull Simulator on Apple products,Mobile Games,Games,USD,2016-01-31,165.00,2015-12-14,0.00,failed,0,US,0.00,False,48
130215,1783668556,Skullplax: Your Trophy's Final Resting Place (...,Hardware,Technology,USD,2013-08-01,25000.00,2013-06-17,185.00,canceled,3,US,185.00,False,45
200672,278430224,Seeing Stella,Shorts,Film & Video,USD,2012-01-30,1500.00,2011-12-21,1516.00,successful,54,US,1516.00,True,40
276950,741881795,Burritos Rolled Easy,Food,Food,USD,2014-09-03,15000.00,2014-08-05,5.00,failed,1,US,5.00,False,29
153597,1925858258,Ten Minutes to The New You,Shorts,Film & Video,USD,2014-09-04,14000.00,2014-08-05,508.00,failed,16,US,508.00,False,30


In [287]:
# Получаем статистическую информацию о числовых столбцах

pd.set_option('display.float_format', '{:.2f}'.format)
df[['goal','usd pledged','backers']].describe()

,goal,usd pledged,backers
count,319321.00,319321.00,319321.00
mean,47648.81,7847.85,102.84
std,1146316.33,84684.97,940.39
min,0.01,0.00,0.00
25%,2000.00,25.00,2.00
50%,5000.00,535.00,12.00
75%,15000.00,3575.00,56.00
max,100000000.00,20338986.27,219382.00


In [288]:
# Обрабатываем выбросы, удаляя 1% самых экстремально высоких значений

n1 = df['goal'].count()

df = df[
    (df['usd pledged'] <= df['usd pledged'].quantile(0.99)) &
    (df['goal'] <= df['goal'].quantile(0.99)) &
    (df['backers'] <= df['backers'].quantile(0.99))
]

n2 = df['goal'].count()

print("Убрано ", n1-n2, " выбросов")
df[['goal','usd pledged','backers']].describe()

Убрано  7463  выбросов


,goal,usd pledged,backers
count,311858.00,311858.00,311858.00
mean,16452.74,4044.36,57.11
std,34158.11,9739.83,131.17
min,0.01,0.00,0.00
25%,2000.00,25.00,2.00
50%,5000.00,517.00,12.00
75%,15000.00,3360.00,53.00
max,374484.00,104884.00,1420.00


### Этап 2: Исследовательский анализ

#### Анализ успешности

In [289]:
# 1. Общая доля успешных проектов

print(f"Общая доля успешных проектов: {round(df["ID"][df["state"] == 'successful'].count() / df["ID"].count() * 100,2)}%")

# Вывод: 
# Общая доля успешных проектов равна 34.88%

Общая доля успешных проектов: 34.88%


In [290]:
# 2. Доля успешных проектов по категориям

category_success = df.groupby('main_category', as_index=False).agg(
    total_projects = ('state','count'),
    successful_projects = ('is_successful','sum'),
    goal_median = ('goal','median'),
    pledged_median = ('usd pledged','median')
)

category_success['success_rate_%'] = (category_success['successful_projects'] / category_success['total_projects'])*100

category_success.sort_values('success_rate_%',ascending=False)

# Вывод: 
# Топ-3 наиболее успешных категории: 
# 1."Танцы" 
# 2."Театр" 
# 3."Комиксы"


,main_category,total_projects,successful_projects,goal_median,pledged_median,success_rate_%
3,Dance,3357,2100,3000.00,1710.00,62.56
14,Theater,9884,5975,3000.00,1500.00,60.45
1,Comics,8603,4374,3600.00,1192.00,50.84
10,Music,44126,21585,4000.00,906.00,48.92
0,Art,23772,9608,3000.00,391.00,40.42
6,Film & Video,55552,21048,6000.00,709.17,37.89
8,Games,26357,8201,8000.00,815.00,31.12
4,Design,22607,6952,10000.00,1310.00,30.75
12,Publishing,33539,10153,5000.00,232.00,30.27
11,Photography,9629,2888,3900.00,201.00,29.99


In [291]:
# 3. Доля успешных проектов по странам

country_success = df.groupby('country', as_index=False).agg(
    total_projects = ('state','count'),
    successful_projects = ('is_successful','sum')
)

country_success['success_rate_%'] = (country_success['successful_projects'] / country_success['total_projects'])*100

country_success.sort_values('success_rate_%',ascending=False)

# Вывод: 
# Топ-3 наиболее успешных стран по сбору средств: 
# 1. США 
# 2. Великобритания 
# 3. Люксембург 

,country,total_projects,successful_projects,success_rate_%
20,US,251721,92216,36.63
9,GB,27018,9045,33.48
13,LU,40,13,32.50
6,DK,751,223,29.69
17,NZ,1119,326,29.13
18,SE,1118,322,28.80
19,SG,117,31,26.50
8,FR,1835,474,25.83
3,CA,11708,2941,25.12
11,IE,560,130,23.21


In [ ]:
# Анализ "качества" спонсоров по категориям

df_backer_q = df[['main_category','usd pledged', 'backers']]
df_backer_q = df_backer_q.groupby('main_category', as_index=False).agg(
    avg_backers = ('backers', 'mean'),
    avg_pledged = ('usd pledged', 'mean'),
)
df_backer_q['pledged_per_backer'] = (df_backer_q['avg_pledged'] / df_backer_q['avg_backers']).fillna(0)

df_backer_q = df_backer_q.sort_values('pledged_per_backer', ascending=False)
df_backer_q

# Вывод:
# В категориях "Фильмы и видео" и "Технологии" спонсоры готовы вкладывать наибольшее кол-во средств, в данных категориях приходится более $90 на одного спонсора.
# В категориях "Игры" и "Комиксы" спонсор, в среднем, готов вложить не более $50, что является наименьшим показателем из всех категорий.
# Стоит также отметить пропорциональное изменение среднего кол-ва спонсоров между высшей и низшей категорией данного рейтинга, 
# этот факт приблизительно уравновешивает итоговое среднее кол-во привлеченных средств. 

,main_category,avg_backers,avg_pledged,pledged_per_backer
6,Film & Video,49.07,4607.33,93.90
13,Technology,59.30,5466.61,92.19
7,Food,44.71,3804.70,85.10
5,Fashion,40.32,3405.76,84.48
14,Theater,45.00,3587.06,79.71
3,Dance,42.18,3234.23,76.67
11,Photography,34.38,2573.13,74.85
4,Design,100.83,7203.94,71.45
10,Music,48.68,3376.15,69.35
0,Art,35.64,2439.87,68.46


#### Анализ длительности кампании

In [293]:
df.sample(5)

,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged,is_successful,duration
287127,80387963,Double Digits: The Story of a Neighborhood Mov...,Documentary,Film & Video,USD,2014-04-12,25000.00,2014-03-13,31786.99,successful,283,US,31786.99,True,30
150442,1906222835,Life Is Beautiful,Photography,Photography,USD,2015-03-25,45.00,2015-03-23,0.00,failed,0,US,0.00,False,2
169202,2020858639,Brimskins (Canceled),Fashion,Fashion,USD,2013-10-09,20000.00,2013-09-03,2340.00,canceled,42,US,2340.00,False,36
25898,1155601051,Help PM bring back great music!!,Music,Music,USD,2016-10-18,30000.00,2016-09-18,31.00,canceled,1,US,31.00,False,30
85182,1512498994,Pirate Attack Card Game (Canceled),Tabletop Games,Games,USD,2013-04-22,20000.00,2013-04-02,440.00,canceled,7,US,440.00,False,20


In [294]:
# 1. Наиболее часто встречающиеся длительности кампаний

duration_count = df.groupby('duration').agg(
    Count = ('ID','count')
)
duration_count["Ratio_%"] = (duration_count["Count"] / df["ID"].count()) * 100
duration_count.sort_values('Count',ascending=False).head(10)

# Вывод:
# Наиболее популярная длительность кампании - 30 дней

,Count,Ratio_%
duration,,
30,139528,44.74
60,27360,8.77
45,14556,4.67
31,10893,3.49
40,8200,2.63
35,8099,2.60
32,6019,1.93
20,5511,1.77
21,5487,1.76


In [ ]:
# 2. Связь длительности кампании и её успеха

duration_df = df[df["state"].isin(['successful','failed','canceled'])]

duration_df['duration_range'] = pd.cut(
    duration_df['duration'],
    bins=[0,30,60,90,120],
    labels=['1-30 days','31-60 days','61-90 days', '91+ days']
    )

duration_df_grouped = duration_df.groupby('duration_range').agg(
    Count = ('ID','count'),
    Success_Count = ('is_successful','sum')
    )
duration_df_grouped['Duration_Ratio_%'] = (duration_df_grouped['Count'] / duration_df["ID"].count())*100
duration_df_grouped['Success_Ratio_In_Range_%'] = (duration_df_grouped['Success_Count'] / duration_df_grouped["Count"])*100


duration_df_grouped

# Вывод:
# Наиболее успешный диапазоны длительности кампании - 1-30 дней

,Count,Success_Count,Duration_Ratio_%,Success_Ratio_In_Range_%
duration_range,,,,
1-30 days,192816,70167,62.99,36.39
31-60 days,107917,36772,35.25,34.07
61-90 days,4907,1706,1.60,34.77
91+ days,483,141,0.16,29.19


#### Корреляционный анализ

In [ ]:
# Кореляционный анализ успешности проекта методом Пирсона функцией .corr

df_corr = df[['is_successful', 'goal', 'usd pledged', 'duration','backers']]
df_corr['pledged_per_backer'] = (df['usd pledged'] / df['backers']).fillna(0)
df_corr_matrix = df_corr.corr()
df_corr_matrix = df_corr_matrix['is_successful'].sort_values(ascending=False)
df_corr_matrix

# Вывод: 
# Наиболее сильные предикторы успеха: 
# 1.Кол-во спонсоров 
# 2.Привлеченные средства 
# Также наблюдается умеренная отрицательная связь с величиной цели сбора, чем выше цель, тем ниже вероятность успеха. 

is_successful         1.00
backers               0.41
usd pledged           0.39
pledged_per_backer    0.14
duration             -0.11
goal                 -0.20
Name: is_successful, dtype: float64